### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 3: Neural networks. Exercise 3**

**A neural network with TensorFlow and PyTorch**

We classify cells as tumour or normal from the expression of two genes, with a small neural network.
We do not look inside: the libraries do the matrix multiplications, the activations, the loss and the gradient descent for us. The goal is to see how to build and train a network in TensorFlow and in PyTorch.

**The maths**

The network has a hidden layer of 8 neurons with ReLU, and one output neuron with a sigmoid that gives the probability of being a tumour cell. Each layer is a matrix multiplication plus a bias, followed by an activation:

$$\hat{y} = \sigma\left( W_2 \, \text{ReLU}(W_1 x + b_1) + b_2 \right)$$

Training finds the weights $W_1, b_1, W_2, b_2$ that minimize the binary cross-entropy, using gradient descent.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import torch

# Check installation
print("All libraries imported successfully")
print(np.__version__)
print(tf.__version__)
print(torch.__version__)

In [ ]:
# Generate example input
# Rows represent cells, columns represent two genes. Label 1 = tumour, 0 = normal
np.random.seed(0)
normal = np.random.normal([2, 2], 0.8, size = (100, 2))
tumour = np.random.normal([4, 4], 0.8, size = (100, 2))
X = np.vstack([normal, tumour])
y = np.array([0] * 100 + [1] * 100)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Format: ", type(X))

In [ ]:
# Plot the cells
plt.figure(figsize = (8, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color = 'blue', label = 'Normal')
plt.scatter(X[y == 1, 0], X[y == 1, 1], color = 'red', label = 'Tumour')
plt.xlabel('gene 1')
plt.ylabel('gene 2')
plt.title('Cells')
plt.legend()
plt.show()

In [ ]:
# Split into training and test cells
# We train on 150 cells and check the accuracy on 50 cells the network has never seen
order = np.random.permutation(len(X))
train, test = order[:150], order[150:]
X_train, y_train = X[train], y[train]
X_test, y_test = X[test], y[test]
print("Training cells:", X_train.shape[0])
print("Test cells:", X_test.shape[0])

In [ ]:
# TensorFlow implementation
tf.random.set_seed(0)

# Build the network: 2 inputs -> 8 neurons (ReLU) -> 1 neuron (sigmoid)
model_tf = tf.keras.Sequential([
    tf.keras.Input(shape = (2,)),
    tf.keras.layers.Dense(8, activation = 'relu'),
    tf.keras.layers.Dense(1, activation = 'sigmoid')])

# Choose the loss and the optimizer (a version of gradient descent)
optimizer_tf = tf.keras.optimizers.Adam(learning_rate = 0.01)
model_tf.compile(optimizer = optimizer_tf, loss = 'binary_crossentropy', metrics = ['accuracy'])
model_tf.summary()

In [ ]:
# Train with TensorFlow
history = model_tf.fit(X_train, y_train, epochs = 100, verbose = 0)
loss_tf = history.history['loss']
print(f"Final training loss: {loss_tf[-1]:.3f}")

# Accuracy on the test cells
test_loss, test_accuracy = model_tf.evaluate(X_test, y_test, verbose = 0)
print(f"TensorFlow test accuracy: {test_accuracy:.2f}")

In [ ]:
# PyTorch implementation
torch.manual_seed(0)

# Build the network: 2 inputs -> 8 neurons (ReLU) -> 1 neuron (sigmoid)
model_torch = torch.nn.Sequential(
    torch.nn.Linear(2, 8),
    torch.nn.ReLU(),
    torch.nn.Linear(8, 1),
    torch.nn.Sigmoid())

# Choose the loss and the optimizer
loss_function = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model_torch.parameters(), lr = 0.01)
print(model_torch)

In [ ]:
# Train with PyTorch
# PyTorch asks us to write the training loop: predict, compute loss, compute gradients, update
X_train_torch = torch.tensor(X_train, dtype = torch.float32)
y_train_torch = torch.tensor(y_train, dtype = torch.float32).reshape(-1, 1)

loss_torch = []
for epoch in range(100):
    y_pred = model_torch(X_train_torch)        # predict
    loss = loss_function(y_pred, y_train_torch) # compute loss
    optimizer.zero_grad()
    loss.backward()                              # compute gradients
    optimizer.step()                             # update weights
    loss_torch.append(loss.item())
print(f"Final training loss: {loss_torch[-1]:.3f}")

# Accuracy on the test cells
X_test_torch = torch.tensor(X_test, dtype = torch.float32)
with torch.no_grad():
    probs = model_torch(X_test_torch).numpy().flatten()
test_accuracy_torch = np.mean((probs > 0.5) == y_test)
print(f"PyTorch test accuracy: {test_accuracy_torch:.2f}")

In [ ]:
# Plot the training loss of both networks
plt.figure(figsize = (8, 5))
plt.plot(loss_tf, label = 'TensorFlow')
plt.plot(loss_torch, label = 'PyTorch')
plt.xlabel('epoch')
plt.ylabel('binary cross-entropy')
plt.title('Training loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Predict a new cell with both networks
new_cell = np.array([[3.5, 3.8]])
p_tf = model_tf.predict(new_cell, verbose = 0)[0, 0]
p_torch = model_torch(torch.tensor(new_cell, dtype = torch.float32)).item()
print("New cell:", new_cell)
print(f"Probability of tumour (TensorFlow): {p_tf:.2f}")
print(f"Probability of tumour (PyTorch):    {p_torch:.2f}")